[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 96. The Resilient Network Design (Robust Optimization)
## Tier 4: The AI/ML/RL Augmentation Method (Neural Architecture Search)

### Key assumptions
- Network topology can be treated as a searchable architectural space
- Neural Architecture Search (NAS) can automatically discover optimal network structures
- Reinforcement learning can guide the search process efficiently
- Multi-objective evaluation balances resilience, cost, and connectivity
- LSTM controllers can generate promising network configurations

### Approach (step-by-step)
1. **Define network architecture space** with variable nodes and connections
2. **Implement LSTM controller** to generate network topologies
3. **Create evaluation function** for resilience, cost, and connectivity metrics
4. **Train controller using REINFORCE** policy gradient method
5. **Iteratively improve architectures** through reinforcement learning
6. **Select best architecture** based on multi-objective fitness

### What to look for in the results
- Learning progress of the NAS controller over episodes
- Discovered network topologies and their performance metrics
- Comparison with traditional optimization methods
- Trade-offs between different objectives (resilience vs cost vs connectivity)

### Concrete example (from the source)
Neural Architecture Search for resilient network design:
- Controller: LSTM with 128 units, 3 layers
- Search space: Networks with 2-4 active nodes, variable connectivity
- Training: 20 episodes, 5 architectures per episode
- Evaluation: Multi-objective fitness combining resilience, cost, and connectivity

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Set, Dict, Optional
import random
import time
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define data structures for Neural Architecture Search
@dataclass
class Facility:
    """Represents a potential facility location"""
    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float

    def distance_to(self, other) -> float:
        """Calculate Euclidean distance to another point"""
        if hasattr(other, 'x') and hasattr(other, 'y'):
            return np.sqrt((self.x - other.x)**2 + (self.y - other.y)**2)
        return 0

@dataclass
class DemandZone:
    """Represents a demand zone"""
    id: int
    x: float
    y: float
    demand: float

@dataclass
class Scenario:
    """Represents a disruption scenario"""
    id: int
    probability: float
    facility_failures: List[int]
    route_blocks: Set[Tuple[int, int]]
    demand_multipliers: Dict[int, float]

@dataclass
class NetworkArchitecture:
    """Represents a network architecture generated by NAS"""
    node_activations: np.ndarray  # Binary vector for active nodes
    adjacency_matrix: np.ndarray  # Connectivity matrix
    num_active_nodes: int
    connectivity_ratio: float

@dataclass
class ArchitecturePerformance:
    """Performance metrics for a network architecture"""
    fitness: float
    resilience: float
    cost: float
    connectivity: float
    service_levels: List[float]

@dataclass
class NASResult:
    """Results from Neural Architecture Search"""
    best_architecture: NetworkArchitecture
    best_performance: ArchitecturePerformance
    training_history: List[Dict]
    execution_time: float

In [ ]:
# Create the problem instance
def create_nas_problem():
    """Create the same problem instance as previous tiers for fair comparison"""

    # Define 8 potential facilities
    facilities = [
        Facility(1, 10, 20, 1000, 200000),   # Location 1
        Facility(2, 30, 15, 1200, 250000),   # Location 2
        Facility(3, 25, 35, 1100, 230000),   # Location 3
        Facility(4, 45, 30, 1300, 280000),   # Location 4
        Facility(5, 15, 50, 900, 180000),    # Location 5
        Facility(6, 40, 10, 1400, 300000),   # Location 6
        Facility(7, 35, 45, 1150, 240000),   # Location 7
        Facility(8, 20, 40, 1050, 210000),    # Location 8
    ]

    # Define 4 demand zones
    demand_zones = [
        DemandZone(1, 15, 25, 200),  # Zone 1
        DemandZone(2, 35, 20, 300),  # Zone 2
        DemandZone(3, 25, 40, 250),  # Zone 3
        DemandZone(4, 40, 35, 350),  # Zone 4
    ]

    # Define disruption scenarios
    scenarios = [
        Scenario(
            id=1,
            probability=0.6,
            facility_failures=[],
            route_blocks=set(),
            demand_multipliers={1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}
        ),
        Scenario(
            id=2,
            probability=0.25,
            facility_failures=[2, 6],
            route_blocks={(1, 3), (4, 2)},
            demand_multipliers={1: 1.5, 2: 0.7, 3: 1.3, 4: 0.8}
        ),
        Scenario(
            id=3,
            probability=0.15,
            facility_failures=[5],
            route_blocks={(3, 1), (7, 4)},
            demand_multipliers={1: 0.8, 2: 1.2, 3: 0.9, 4: 1.1}
        )
    ]

    return facilities, demand_zones, scenarios

# Create problem instance
facilities, demand_zones, scenarios = create_nas_problem()

print("NAS Problem Setup:")
print(f"- Facilities: {len(facilities)} potential locations")
print(f"- Demand zones: {len(demand_zones)} service areas")
print(f"- Scenarios: {len(scenarios)} disruption scenarios")
print(f"- Objective: Discover optimal network topology using NAS")

NAS Problem Setup:
- Facilities: 8 potential locations
- Demand zones: 4 service areas
- Scenarios: 3 disruption scenarios
- Objective: Discover optimal network topology using NAS


In [ ]:
# Implement the Neural Architecture Search framework
class NetworkArchitectureController:
    """LSTM-based controller for generating network architectures"""

    def __init__(self, max_nodes: int = 8, max_connections: int = 20,
                 hidden_units: int = 128, learning_rate: float = 0.01):
        self.max_nodes = max_nodes
        self.max_connections = max_connections
        self.hidden_units = hidden_units
        self.learning_rate = learning_rate

        # Initialize LSTM weights (simplified implementation)
        self.weights = {
            'Wxh': np.random.randn(hidden_units, max_nodes + max_connections) * 0.1,
            'Whh': np.random.randn(hidden_units, hidden_units) * 0.1,
            'Why': np.random.randn(max_nodes + max_connections, hidden_units) * 0.1,
            'bh': np.zeros(hidden_units),
            'by': np.zeros(max_nodes + max_connections)
        }

        # Baseline for REINFORCE algorithm
        self.baseline = 0.0

        # Training history
        self.training_history = []

    def lstm_forward(self, x_seq, h_prev=None):
        """Simplified LSTM forward pass"""
        if h_prev is None:
            h_prev = np.zeros(self.hidden_units)

        outputs = []
        h = h_prev.copy()

        for x in x_seq:
            # Simplified LSTM computation
            h = np.tanh(np.dot(self.weights['Wxh'], x) +
                       np.dot(self.weights['Whh'], h) + self.weights['bh'])
            y = np.dot(self.weights['Why'], h) + self.weights['by']
            outputs.append(y)

        return outputs, h

    def generate_architecture(self) -> NetworkArchitecture:
        """Generate a network architecture using the LSTM controller"""
        # Generate random input sequence
        input_seq = [np.random.randn(self.max_nodes + max_connections) for _ in range(3)]

        # Forward pass through LSTM
        outputs, _ = self.lstm_forward(input_seq)

        # Use last output to generate architecture parameters
        arch_params = outputs[-1]

        # Decode node activations (sigmoid for binary decisions)
        node_probs = 1 / (1 + np.exp(-arch_params[:self.max_nodes]))
        node_activations = (node_probs > 0.5).astype(int)

        # Ensure at least 2 active nodes
        if np.sum(node_activations) < 2:
            # Activate the 2 nodes with highest probabilities
            top_indices = np.argsort(node_probs)[-2:]
            node_activations[top_indices] = 1

        # Decode connections (sigmoid for binary decisions)
        connection_probs = 1 / (1 + np.exp(-arch_params[self.max_nodes:]))

        # Build adjacency matrix for active nodes
        num_active = np.sum(node_activations)
        adjacency = np.zeros((num_active, num_active))

        if num_active >= 2:
            # Map connection probabilities to adjacency matrix
            active_indices = np.where(node_activations == 1)[0]
            conn_idx = 0

            for i in range(num_active):
                for j in range(i + 1, num_active):
                    if conn_idx < len(connection_probs):
                        if connection_probs[conn_idx] > 0.3:  # Threshold for connections
                            adjacency[i, j] = adjacency[j, i] = 1
                        conn_idx += 1

        # Calculate connectivity ratio
        possible_connections = num_active * (num_active - 1) / 2 if num_active >= 2 else 1
        actual_connections = np.sum(adjacency) / 2
        connectivity_ratio = actual_connections / possible_connections if possible_connections > 0 else 0

        return NetworkArchitecture(
            node_activations=node_activations,
            adjacency_matrix=adjacency,
            num_active_nodes=num_active,
            connectivity_ratio=connectivity_ratio
        )

    def update_weights(self, reward, architecture, learning_rate=None):
        """Update weights using REINFORCE algorithm (simplified)"""
        if learning_rate is None:
            learning_rate = self.learning_rate

        # Update baseline
        self.baseline = 0.9 * self.baseline + 0.1 * reward

        # Simplified REINFORCE update (in practice, this would involve gradient computation)
        # For demonstration, we'll add small random updates
        for key in self.weights:
            if key.startswith('W') or key.startswith('b'):
                # Add small random noise to simulate gradient updates
                noise = np.random.randn(*self.weights[key].shape) * learning_rate * 0.01
                self.weights[key] += noise * (reward - self.baseline)

class ResilientNetworkNAS:
    """Neural Architecture Search for resilient network design"""

    def __init__(self, facilities: List[Facility], demand_zones: List[DemandZone],
                 scenarios: List[Scenario]):
        self.facilities = facilities
        self.demand_zones = demand_zones
        self.scenarios = scenarios

        # Initialize controller
        self.controller = NetworkArchitectureController(
            max_nodes=len(facilities),
            max_connections=len(facilities) * (len(facilities) - 1) // 2
        )

        # Training history
        self.training_history = []
        self.best_architecture = None
        self.best_performance = None

    def evaluate_architecture(self, architecture: NetworkArchitecture) -> ArchitecturePerformance:
        """Evaluate a network architecture"""
        # Select facilities based on architecture
        active_facilities = []
        for i, active in enumerate(architecture.node_activations):
            if active and i < len(self.facilities):
                active_facilities.append(self.facilities[i])

        if len(active_facilities) < 2:
            return ArchitecturePerformance(
                fitness=float('inf'),
                resilience=0.0,
                cost=float('inf'),
                connectivity=0.0,
                service_levels=[]
            )

        # Evaluate across scenarios
        total_fitness = 0
        resilience_scores = []
        total_cost = 0
        service_levels = []

        for scenario in self.scenarios:
            fitness, resilience, cost = self._evaluate_scenario(
                active_facilities, architecture, scenario
            )

            total_fitness += scenario.probability * fitness
            resilience_scores.append(resilience)
            total_cost += scenario.probability * cost
            service_levels.append(resilience)  # Use resilience as service level proxy

        avg_resilience = np.mean(resilience_scores)
        connectivity = architecture.connectivity_ratio

        # Multi-objective fitness function
        fitness_score = (avg_resilience * 1000 - total_cost / 1e6 + connectivity * 100)

        return ArchitecturePerformance(
            fitness=fitness_score,
            resilience=min(resilience_scores),  # Worst-case resilience
            cost=total_cost,
            connectivity=connectivity,
            service_levels=service_levels
        )

    def _evaluate_scenario(self, facilities: List[Facility],
                           architecture: NetworkArchitecture, scenario: Scenario) -> Tuple[float, float, float]:
        """Evaluate architecture for a specific scenario"""
        # Filter available facilities
        available_facilities = [f for f in facilities
                               if f.id not in scenario.facility_failures]

        if len(available_facilities) < 1:
            return 0, 0, float('inf')

        # Calculate connectivity resilience
        connectivity_resilience = self._calculate_connectivity(
            available_facilities, architecture, scenario
        )

        # Calculate demand coverage
        service_coverage = self._calculate_coverage(available_facilities, scenario)

        # Calculate costs
        fixed_costs = sum(f.fixed_cost for f in available_facilities)
        transport_costs = self._calculate_transport(available_facilities, scenario)

        fitness = (connectivity_resilience + service_coverage) / 2
        resilience = min(connectivity_resilience, service_coverage)

        return fitness, resilience, fixed_costs + transport_costs

    def _calculate_connectivity(self, facilities: List[Facility],
                               architecture: NetworkArchitecture, scenario: Scenario) -> float:
        """Calculate connectivity resilience"""
        n = len(facilities)
        if n < 2:
            return 0

        # Use architecture's adjacency matrix (simplified)
        current_adj = architecture.adjacency_matrix[:n, :n] if n <= architecture.num_active_nodes else np.eye(n)

        # Remove blocked routes (simplified)
        for route in scenario.route_blocks:
            # This is simplified - in practice would map routes to adjacency matrix indices
            pass

        # Connectivity as fraction of possible connections
        possible = n * (n - 1) / 2
        actual = np.sum(current_adj) / 2
        return actual / possible if possible > 0 else 0

    def _calculate_coverage(self, facilities: List[Facility], scenario: Scenario) -> float:
        """Calculate demand coverage"""
        total_demand = 0
        covered = 0

        for i, demand in enumerate(self.demand_zones):
            actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
            total_demand += actual_demand

            best_service = 0
            for facility in facilities:
                if (facility.id, demand.id) not in scenario.route_blocks:
                    distance = facility.distance_to(demand)
                    service = min(facility.capacity, actual_demand) * np.exp(-distance / 1000)
                    best_service = max(best_service, service)

            covered += min(best_service, actual_demand)

        avg_multiplier = np.mean([scenario.demand_multipliers.get(d.id, 1.0) for d in self.demand_zones])
        return covered / (total_demand * avg_multiplier) if total_demand > 0 else 0

    def _calculate_transport(self, facilities: List[Facility], scenario: Scenario) -> float:
        """Calculate transport costs"""
        total_cost = 0

        for i, demand in enumerate(self.demand_zones):
            actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
            min_cost = float('inf')

            for facility in facilities:
                if (facility.id, demand.id) not in scenario.route_blocks:
                    distance = facility.distance_to(demand)
                    cost = distance * 0.01 * actual_demand
                    min_cost = min(min_cost, cost)

            if min_cost < float('inf'):
                total_cost += min_cost

        return total_cost

    def train_controller(self, num_episodes: int = 20, architectures_per_episode: int = 5) -> NASResult:
        """Train the NAS controller using REINFORCE"""
        start_time = time.time()

        print("=" * 60)
        print("NEURAL ARCHITECTURE SEARCH TRAINING")
        print("=" * 60)
        print(f"Episodes: {num_episodes}")
        print(f"Architectures per episode: {architectures_per_episode}")

        for episode in range(num_episodes):
            episode_architectures = []
            episode_performances = []
            episode_rewards = []

            # Generate architectures for this episode
            for _ in range(architectures_per_episode):
                architecture = self.controller.generate_architecture()
                performance = self.evaluate_architecture(architecture)

                episode_architectures.append(architecture)
                episode_performances.append(performance)
                episode_rewards.append(performance.fitness)

            # Update baseline
            episode_mean = np.mean(episode_rewards)
            self.controller.baseline = (0.9 * self.controller.baseline + 0.1 * episode_mean
                                      if self.controller.baseline != 0 else episode_mean)

            # Store episode results
            episode_data = {
                'episode': episode,
                'mean_reward': episode_mean,
                'best_reward': max(episode_rewards),
                'baseline': self.controller.baseline,
                'architectures': episode_architectures,
                'performances': episode_performances
            }
            self.training_history.append(episode_data)

            # Update controller weights (simplified REINFORCE)
            for architecture, performance in zip(episode_architectures, episode_performances):
                self.controller.update_weights(performance.fitness, architecture)

            # Update best architecture
            best_idx = np.argmax(episode_rewards)
            if (self.best_performance is None or
                episode_performances[best_idx].fitness > self.best_performance.fitness):
                self.best_architecture = episode_architectures[best_idx]
                self.best_performance = episode_performances[best_idx]

            # Progress reporting
            if episode % 5 == 0:
                print(f"Episode {episode:2d}: Mean={episode_mean:.2f}, "
                      f"Best={max(episode_rewards):.2f}, "
                      f"Baseline={self.controller.baseline:.2f}")

        execution_time = time.time() - start_time

        return NASResult(
            best_architecture=self.best_architecture,
            best_performance=self.best_performance,
            training_history=self.training_history,
            execution_time=execution_time
        )

In [ ]:
try:
    # Run Neural Architecture Search
    def run_nas():
        """Execute NAS and analyze results"""
    
        # Initialize NAS
        nas = ResilientNetworkNAS(facilities, demand_zones, scenarios)
    
        # Train controller
        result = nas.train_controller(num_episodes=20, architectures_per_episode=5)
    
        # Analyze results
        print("\n" + "=" * 60)
        print("NAS TRAINING RESULTS")
        print("=" * 60)
    
        print(f"\nBest Architecture:")
        print(f"  Active Nodes: {result.best_architecture.num_active_nodes}")
        print(f"  Connectivity Ratio: {result.best_architecture.connectivity_ratio:.3f}")
        print(f"  Node Activations: {result.best_architecture.node_activations}")
    
        print(f"\nPerformance Metrics:")
        print(f"  Fitness Score: {result.best_performance.fitness:.2f}")
        print(f"  Resilience: {result.best_performance.resilience:.3f}")
        print(f"  Total Cost: ${result.best_performance.cost:,.0f}")
        print(f"  Connectivity: {result.best_performance.connectivity:.3f}")
        print(f"  Execution Time: {result.execution_time:.2f} seconds")
    
        # Selected facilities
        selected_facilities = []
        for i, active in enumerate(result.best_architecture.node_activations):
            if active and i < len(facilities):
                selected_facilities.append(facilities[i])
    
        print(f"\nSelected Facilities:")
        for facility in selected_facilities:
            print(f"  Facility {facility.id}: ({facility.x}, {facility.y}) - Cost: ${facility.fixed_cost:,}")
    
        # Check service requirements
        print(f"\nService Analysis:")
        print(f"  Service Requirement: {'✓' if result.best_performance.resilience >= 0.8 else '✗'}")
        print(f"  Worst-Case Coverage: {result.best_performance.resilience:.1%}")
        print(f"  Service Levels by Scenario: {[f'{s:.1%}' for s in result.best_performance.service_levels]}")
    
        return nas, result
    
    # Run NAS
    nas, nas_result = run_nas()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'max_connections' is not defined...]

In [ ]:
try:
    # Create comprehensive NAS visualization
    def create_nas_visualization():
        """Create comprehensive visualization of NAS results"""
    
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Neural Architecture Search for Resilient Network Design', fontsize=16, fontweight='bold')
    
        # Panel 1: Training Progress
        ax1 = axes[0, 0]
        ax1.set_title('NAS Training Progress', fontweight='bold')
    
        episodes = range(len(nas_result.training_history))
        mean_rewards = [ep['mean_reward'] for ep in nas_result.training_history]
        best_rewards = [ep['best_reward'] for ep in nas_result.training_history]
        baselines = [ep['baseline'] for ep in nas_result.training_history]
    
        ax1.plot(episodes, mean_rewards, 'b-', linewidth=2, marker='o', markersize=4, label='Mean Reward')
        ax1.plot(episodes, best_rewards, 'g-', linewidth=2, marker='s', markersize=4, label='Best Reward')
        ax1.plot(episodes, baselines, 'r--', linewidth=2, label='Baseline')
    
        ax1.set_xlabel('Episode')
        ax1.set_ylabel('Reward')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
        # Panel 2: Architecture Visualization
        ax2 = axes[0, 1]
        ax2.set_title('Best Network Architecture', fontweight='bold')
    
        # Plot selected facilities
        selected_facilities = []
        for i, active in enumerate(nas_result.best_architecture.node_activations):
            if active and i < len(facilities):
                selected_facilities.append(facilities[i])
    
        # Plot all facilities
        for i, facility in enumerate(facilities):
            color = 'green' if facility in selected_facilities else 'lightgray'
            marker = 'o' if facility in selected_facilities else 'x'
            size = 300 if facility in selected_facilities else 200
            ax2.scatter(facility.x, facility.y, c=color, marker=marker, s=size,
                       alpha=0.7, edgecolors='black', linewidth=2)
            ax2.annotate(f'F{facility.id}', (facility.x, facility.y),
                        xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
        # Draw connections based on adjacency matrix
        if len(selected_facilities) >= 2:
            for i in range(len(selected_facilities)):
                for j in range(i + 1, len(selected_facilities)):
                    if (i < nas_result.best_architecture.adjacency_matrix.shape[0] and
                        j < nas_result.best_architecture.adjacency_matrix.shape[1] and
                        nas_result.best_architecture.adjacency_matrix[i, j] > 0):
                        ax2.plot([selected_facilities[i].x, selected_facilities[j].x],
                               [selected_facilities[i].y, selected_facilities[j].y],
                               'k-', alpha=0.5, linewidth=2)
    
        # Plot demand zones
        for demand in demand_zones:
            ax2.scatter(demand.x, demand.y, c='blue', marker='s', s=200,
                       alpha=0.7, edgecolors='black', linewidth=2)
            ax2.annotate(f'D{demand.id}', (demand.x, demand.y),
                        xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
        ax2.set_xlabel('X Coordinate')
        ax2.set_ylabel('Y Coordinate')
        ax2.grid(True, alpha=0.3)
        ax2.legend(['Selected Facility', 'Rejected Facility', 'Connection', 'Demand Zone'], loc='upper right')
    
        # Panel 3: Performance Metrics
        ax3 = axes[1, 0]
        ax3.set_title('Multi-Objective Performance', fontweight='bold')
    
        metrics = ['Resilience', 'Cost (M$)', 'Connectivity', 'Fitness']
        values = [
            nas_result.best_performance.resilience,
            nas_result.best_performance.cost / 1e6,
            nas_result.best_performance.connectivity,
            nas_result.best_performance.fitness / 1000  # Scale for visibility
        ]
    
        bars = ax3.bar(metrics, values, alpha=0.8, color=['green', 'red', 'blue', 'orange'])
        ax3.set_ylabel('Value')
        ax3.grid(True, alpha=0.3)
    
        # Add value labels on bars
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                    f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
        # Panel 4: Architecture Diversity
        ax4 = axes[1, 1]
        ax4.set_title('Architecture Diversity Over Training', fontweight='bold')
    
        # Calculate diversity metrics
        diversity_scores = []
        for episode_data in nas_result.training_history:
            architectures = episode_data['architectures']
            if len(architectures) > 1:
                # Calculate pairwise diversity
                total_diversity = 0
                count = 0
                for i in range(len(architectures)):
                    for j in range(i + 1, len(architectures)):
                        # Diversity based on node activation difference
                        diff = np.sum(np.abs(architectures[i].node_activations -
                                            architectures[j].node_activations))
                        total_diversity += diff
                        count += 1
                diversity_scores.append(total_diversity / count if count > 0 else 0)
            else:
                diversity_scores.append(0)
    
        episodes = range(len(diversity_scores))
        ax4.plot(episodes, diversity_scores, 'mo-', linewidth=2, markersize=6)
        ax4.set_xlabel('Episode')
        ax4.set_ylabel('Architecture Diversity')
        ax4.grid(True, alpha=0.3)
    
        plt.tight_layout()
        plt.show()
    
        return fig
    
    # Create visualization
    fig = create_nas_visualization()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'nas_result' is not defined...]

In [ ]:
try:
    # Performance comparison with previous tiers
    def compare_nas_with_other_methods():
        """Compare NAS performance with traditional optimization methods"""
    
        print("\n" + "=" * 60)
        print("NAS vs TRADITIONAL METHODS COMPARISON")
        print("=" * 60)
    
        # NAS results summary
        print(f"\nNeural Architecture Search Results:")
        print(f"  Best Fitness: {nas_result.best_performance.fitness:.2f}")
        print(f"  Resilience: {nas_result.best_performance.resilience:.1%}")
        print(f"  Total Cost: ${nas_result.best_performance.cost:,.0f}")
        print(f"  Connectivity: {nas_result.best_performance.connectivity:.3f}")
        print(f"  Training Time: {nas_result.execution_time:.2f} seconds")
        print(f"  Episodes Trained: {len(nas_result.training_history)}")
    
        # Method characteristics comparison
        print(f"\nMethod Characteristics:")
        print(f"\nTier 1 (Mathematical Optimization):")
        print(f"  ✓ Provable optimality")
        print(f"  ✓ Exact constraint satisfaction")
        print(f"  ✗ Exponential time complexity")
        print(f"  ✗ Limited to small problems")
    
        print(f"\nTier 2 (Sweep Algorithm):")
        print(f"  ✓ Polynomial time complexity")
        print(f"  ✓ Geographical diversity focus")
        print(f"  ✓ Simple implementation")
        print(f"  ✗ Limited search patterns")
    
        print(f"\nTier 3 (Grasshopper Optimization):")
        print(f"  ✓ Population-based search")
        print(f"  ✓ Global optimization capability")
        print(f"  ✓ Adaptive behavior")
        print(f"  ⚠ Parameter sensitivity")
    
        print(f"\nTier 4 (Neural Architecture Search):")
        print(f"  ✓ Automated topology discovery")
        print(f"  ✓ Learning-based improvement")
        print(f"  ✓ Multi-objective optimization")
        print(f"  ✓ Adaptive search strategy")
        print(f"  ⚠ Training computational cost")
        print(f"  ⚠ Requires reward function design")
    
        # Learning analysis
        if len(nas_result.training_history) > 1:
            print(f"\nLearning Analysis:")
            initial_reward = nas_result.training_history[0]['mean_reward']
            final_reward = nas_result.training_history[-1]['mean_reward']
            improvement = ((final_reward - initial_reward) / abs(initial_reward)) * 100 if initial_reward != 0 else 0
    
            print(f"  Initial Mean Reward: {initial_reward:.2f}")
            print(f"  Final Mean Reward: {final_reward:.2f}")
            print(f"  Improvement: {improvement:+.1f}%")
    
            # Calculate learning stability
            recent_rewards = [ep['mean_reward'] for ep in nas_result.training_history[-5:]]
            reward_stability = np.std(recent_rewards) / np.mean(recent_rewards) if np.mean(recent_rewards) > 0 else 0
            print(f"  Recent Stability (CV): {reward_stability:.3f} ({'Stable' if reward_stability < 0.1 else 'Unstable'})")
    
        # Architecture analysis
        print(f"\nDiscovered Architecture Analysis:")
        print(f"  Active Nodes: {nas_result.best_architecture.num_active_nodes} / {len(facilities)}")
        print(f"  Connectivity Ratio: {nas_result.best_architecture.connectivity_ratio:.3f}")
        print(f"  Network Density: {'Sparse' if nas_result.best_architecture.connectivity_ratio < 0.3 else 'Medium' if nas_result.best_architecture.connectivity_ratio < 0.6 else 'Dense'}")
    
        # Multi-objective trade-off analysis
        print(f"\nMulti-Objective Trade-offs:")
        resilience_score = nas_result.best_performance.resilience
        cost_score = 1 - (nas_result.best_performance.cost / max(f.fixed_cost for f in facilities))  # Normalized
        connectivity_score = nas_result.best_performance.connectivity
    
        print(f"  Resilience Weight: {resilience_score:.3f}")
        print(f"  Cost Efficiency: {cost_score:.3f}")
        print(f"  Network Connectivity: {connectivity_score:.3f}")
        print(f"  Balance Score: {1 - np.std([resilience_score, cost_score, connectivity_score]):.3f}")
    
        return nas_result
    
    # Compare methods
    comparison_result = compare_nas_with_other_methods()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'nas_result' is not defined...]

In [ ]:
try:
    # Architecture exploration analysis
    def analyze_architecture_exploration():
        """Analyze the exploration of architecture space during NAS training"""
    
        print("\n" + "=" * 60)
        print("ARCHITECTURE SPACE EXPLORATION ANALYSIS")
        print("=" * 60)
    
        # Collect all architectures from training
        all_architectures = []
        all_performances = []
    
        for episode_data in nas_result.training_history:
            all_architectures.extend(episode_data['architectures'])
            all_performances.extend(episode_data['performances'])
    
        print(f"Total architectures explored: {len(all_architectures)}")
    
        # Analyze node activation patterns
        node_activation_counts = np.zeros(len(facilities))
        for arch in all_architectures:
            node_activation_counts += arch.node_activations
    
        print(f"\nFacility Selection Frequency:")
        for i, count in enumerate(node_activation_counts):
            frequency = count / len(all_architectures)
            print(f"  Facility {i+1}: Selected {count}/{len(all_architectures)} times ({frequency:.1%})")
    
        # Analyze connectivity patterns
        connectivity_ratios = [arch.connectivity_ratio for arch in all_architectures]
        print(f"\nConnectivity Analysis:")
        print(f"  Average Connectivity: {np.mean(connectivity_ratios):.3f}")
        print(f"  Connectivity Range: {np.min(connectivity_ratios):.3f} - {np.max(connectivity_ratios):.3f}")
        print(f"  Most Common Connectivity: {max(set(connectivity_ratios), key=list(connectivity_ratios).count):.3f}")
    
        # Performance distribution
        fitness_scores = [perf.fitness for perf in all_performances]
        resilience_scores = [perf.resilience for perf in all_performances]
        cost_scores = [perf.cost for perf in all_performances]
    
        print(f"\nPerformance Distribution:")
        print(f"  Fitness: {np.mean(fitness_scores):.2f} ± {np.std(fitness_scores):.2f}")
        print(f"  Resilience: {np.mean(resilience_scores):.3f} ± {np.std(resilience_scores):.3f}")
        print(f"  Cost: ${np.mean(cost_scores):,.0f} ± ${np.std(cost_scores):,.0f}")
    
        # Create exploration visualization
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('Architecture Space Exploration Analysis', fontsize=16, fontweight='bold')
    
        # Panel 1: Facility Selection Frequency
        ax1 = axes[0, 0]
        ax1.set_title('Facility Selection Frequency', fontweight='bold')
        facility_ids = range(1, len(facilities) + 1)
        frequencies = node_activation_counts / len(all_architectures)
    
        bars = ax1.bar(facility_ids, frequencies, alpha=0.8)
        ax1.set_xlabel('Facility ID')
        ax1.set_ylabel('Selection Frequency')
        ax1.grid(True, alpha=0.3)
    
        # Panel 2: Connectivity Distribution
        ax2 = axes[0, 1]
        ax2.set_title('Connectivity Ratio Distribution', fontweight='bold')
        ax2.hist(connectivity_ratios, bins=15, alpha=0.7, edgecolor='black')
        ax2.set_xlabel('Connectivity Ratio')
        ax2.set_ylabel('Frequency')
        ax2.grid(True, alpha=0.3)
    
        # Panel 3: Performance Scatter Plot
        ax3 = axes[1, 0]
        ax3.set_title('Resilience vs Cost Trade-off', fontweight='bold')
    
        scatter = ax3.scatter(resilience_scores, cost_scores, c=fitness_scores,
                             cmap='viridis', alpha=0.7, s=50)
        ax3.set_xlabel('Resilience')
        ax3.set_ylabel('Cost ($)')
        ax3.grid(True, alpha=0.3)
    
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax3)
        cbar.set_label('Fitness')
    
        # Highlight best solution
        best_idx = np.argmax(fitness_scores)
        ax3.scatter(resilience_scores[best_idx], cost_scores[best_idx],
                   c='red', s=200, marker='*', edgecolors='black', linewidth=2,
                   label='Best Architecture', zorder=5)
        ax3.legend()
    
        # Panel 4: Learning Progress
        ax4 = axes[1, 1]
        ax4.set_title('Learning Progress by Episode', fontweight='bold')
    
        episodes = range(len(nas_result.training_history))
        mean_fitness = [np.mean([p.fitness for p in ep['performances']]) for ep in nas_result.training_history]
        best_fitness = [max([p.fitness for p in ep['performances']]) for ep in nas_result.training_history]
    
        ax4.plot(episodes, mean_fitness, 'b-', linewidth=2, marker='o', markersize=4, label='Mean Fitness')
        ax4.plot(episodes, best_fitness, 'r-', linewidth=2, marker='s', markersize=4, label='Best Fitness')
        ax4.set_xlabel('Episode')
        ax4.set_ylabel('Fitness')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
        plt.tight_layout()
        plt.show()
    
        return all_architectures, all_performances
    
    # Analyze exploration
    all_archs, all_perfs = analyze_architecture_exploration()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'nas_result' is not defined...]

### Why this Tier exists vs earlier Tiers
This Tier 4 provides AI/ML augmentation when traditional optimization methods are insufficient:

**Key Advantages over Tier 1-3:**
- **Automated discovery**: Learns optimal network topologies automatically
- **Adaptive search**: Improves search strategy through learning
- **Multi-objective optimization**: Naturally balances competing objectives
- **Topology flexibility**: Explores diverse network structures beyond predefined patterns
- **Learning-based improvement**: Performance improves with training

**When to prefer this Tier:**
- Complex network design problems with non-obvious optimal structures
- Situations requiring balance between multiple competing objectives
- Problems where traditional heuristics consistently find suboptimal solutions
- Research and development exploring novel network configurations
- Dynamic environments requiring adaptive solution approaches

### Pros / Cons vs earlier Tiers
**Pros:**
- Discovers non-intuitive optimal network topologies
- Automatically balances multiple objectives
- Improves performance through learning
- Flexible search beyond predefined patterns
- Adapts to problem-specific characteristics

**Cons:**
- Higher computational cost for training
- Requires careful reward function design
- No guarantee of finding global optimum
- Performance depends on training quality
- More complex implementation and tuning

### When to use this Tier
- **Complex network design** with non-linear interactions
- **Multi-objective optimization** where trade-offs are critical
- **Research applications** exploring novel network topologies
- **Dynamic environments** requiring adaptive solutions
- **Large-scale problems** where traditional methods struggle
- **Innovation projects** seeking breakthrough network designs